In [ ]:
#下載台北市youbike資料
import requests
response = requests.get('https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json')
response.encoding = 'utf-8'
youbikeData = response.json()
youbikeData

In [ ]:
import psycopg2
import os
conn = psycopg2.connect(host="localhost",database='postgres',user=os.environ["POSTGRES_USER"],password=os.environ["POSTGRES_PASSWORD"])

with conn:
    with conn.cursor() as cursor:
        #insert站點資訊
        for site in youbikeData:
            sql = '''
                INSERT INTO 站點資訊 
                VALUES(%s,%s,%s,%s,%s,%s)
                ON CONFLICT DO NOTHING;
            '''
            #欄位名稱依照目前的開放資料格式:latitude、longitude
            cursor.execute(sql,(site['sno'],site['sna'],site['sarea'],site['ar'],site['latitude'],site['longitude']))

            #插入youbike資訊
            sql = '''
                INSERT INTO youbike
                VALUES(%s,%s,%s,%s,%s,%s)
                ON CONFLICT DO NOTHING;
            '''
            #Quantity:總車輛、available_rent_bikes:可借、available_return_bikes:可還
            #act 是字串 '1' 或 '0',bool('0') 也是 True,所以要用 == '1' 判斷
            cursor.execute(sql,(site['mday'],site['sno'],site['Quantity'],site['available_rent_bikes'],site['available_return_bikes'],site['act'] == '1'))


conn.close()